# Mapa de Medellín: superficie gaussiana y divisiones políticas

Este cuaderno usa `P_gaussiana_regional_9377.tif`, que conserva la superficie completa generada por el análisis. No utiliza el ráster municipal enmascarado, para evitar los escalones blancos en los bordes.

In [ ]:
%pip -q install geopandas rasterio pyogrio matplotlib

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

ROOT = Path('/content/drive/MyDrive/2.Consultoria/01_PAC_MEDELLIN_2026/2.Ejecución/C1_01_EVIDENCIA_DIAGNOSTICO_PROSPECTIVA/P1_Caracterizacion_Socioeconomica_Ambiental/02_DOCUMENTOS_EN_ELABORACION/07_1_CLIMA_RELIEVE')
ANALISIS = ROOT / '06_ANALISIS_PRECIPITACION'
ejecuciones = sorted(p for p in ANALISIS.iterdir() if p.is_dir())
if not ejecuciones:
    raise FileNotFoundError('No hay ejecuciones en 06_ANALISIS_PRECIPITACION.')
OUT = ejecuciones[-1]

archivo_raster = OUT / 'P_gaussiana_regional_9377.tif'
archivo_limites = next(ROOT.glob('03_DATOS_PREPARADOS/*/limites_medellin_9377.gpkg'))

with rasterio.open(archivo_raster) as src:
    raster = src.read(1).astype(float)
    bounds = src.bounds
    crs = src.crs
    if src.nodata is not None:
        raster[raster == src.nodata] = np.nan

limite = gpd.read_file(archivo_limites, layer='limite_medellin').to_crs(crs)
municipio = limite.geometry.union_all()

# Buscar una sola capa de comunas; si no existe, usar barrios.
capas = gpd.list_layers(archivo_limites)['name'].tolist()
capa_politica = next((n for n in capas if 'comuna' in n.lower()), None)
if capa_politica is None:
    capa_politica = next((n for n in capas if 'barrio' in n.lower()), None)
if capa_politica is None:
    raise ValueError('No se encontró una capa de comunas ni de barrios.')

divisiones = gpd.read_file(archivo_limites, layer=capa_politica).to_crs(crs)
divisiones = gpd.clip(divisiones, limite)

valores = raster[np.isfinite(raster)]
norm = Normalize(vmin=np.percentile(valores, 2), vmax=np.percentile(valores, 98))

fig, ax = plt.subplots(figsize=(11, 11))
ax.imshow(raster, extent=[bounds.left, bounds.right, bounds.bottom, bounds.top], origin='upper', cmap='YlGnBu', norm=norm, interpolation='bilinear')

# Límites políticos internos, deliberadamente ligeros.
divisiones.boundary.plot(ax=ax, color='black', linewidth=0.45, alpha=0.55)
limite.boundary.plot(ax=ax, color='#b2182b', linewidth=2.5)

minx, miny, maxx, maxy = municipio.bounds
pad = 1500
ax.set_xlim(minx - pad, maxx + pad)
ax.set_ylim(miny - pad, maxy + pad)
ax.set_aspect('equal')
ax.ticklabel_format(style='plain', useOffset=False)
ax.set_xlabel('Este (m) — MAGNA-SIRGAS / Origen Nacional')
ax.set_ylabel('Norte (m) — MAGNA-SIRGAS / Origen Nacional')
ax.set_title('Precipitación media anual en Medellín\nSuperficie gaussiana y ' + capa_politica, fontsize=15, pad=15)

ax.legend(handles=[
    Line2D([0], [0], color='#b2182b', linewidth=2.5, label='Límite municipal'),
    Line2D([0], [0], color='black', linewidth=0.45, label=capa_politica)
], loc='upper left', fontsize=9, frameon=True)

fig.colorbar(ScalarMappable(norm=norm, cmap='YlGnBu'), ax=ax, shrink=0.72, label='Precipitación media anual (mm/año)')
ax.annotate('N', xy=(0.96, 0.95), xytext=(0.96, 0.86), xycoords='axes fraction', ha='center', fontsize=13, arrowprops={'arrowstyle': '->', 'color': 'black', 'linewidth': 1.2})

salida = OUT / 'mapa_gaussiana_medellin_divisiones_corregido.png'
plt.tight_layout()
plt.savefig(salida, dpi=220, bbox_inches='tight')
plt.show()

print('Ráster utilizado:', archivo_raster)
print('Capa política utilizada:', capa_politica)
print('Mapa guardado en:', salida)